<a href="https://colab.research.google.com/github/shin-noda/leetcode-problemset-97/blob/main/Problem1044_Improved.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
class Solution:
    def longestDupSubstring(self, s):
        # Convert string to integers (plus a 0 terminator for the '$' concept)
        # This makes it easy to handle recursion with renamed alphabet characters.
        A = [ord(ch) - ord('a') + 1 for ch in s] + [0]

        # Run SA-IS to get the full Suffix Array
        suffix_array = self.sais(A, max(A) + 1)

        # The first element of suffix_array is always the virtual terminator we added
        # so we slice it out to match the original string length
        suffix_array = suffix_array[1:]

        # Build LCP array using Kasai's Algorithm (O(N))
        n = len(s)
        lcp = [0] * n
        rank = [0] * n
        for i, sa_val in enumerate(suffix_array):
            rank[sa_val] = i

        h = 0
        max_len = 0
        start_idx = 0

        for i in range(n):
            if rank[i] > 0:
                j = suffix_array[rank[i] - 1]
                while i + h < n and j + h < n and s[i + h] == s[j + h]:
                    h += 1

            lcp[rank[i]] = h

            if h > max_len:
                max_len = h
                start_idx = i

            if h > 0:
                h -= 1

        if max_len <= 0:
            return ""

        return s[start_idx : start_idx + max_len]

    def sais(self, A, alphabet_size):
        n = len(A)

        if n == 0:
            return []

        if n == 1:
            return [0]

        if n == 2:
            if A[0] > A[1]:
                return [1, 0]

            else:
                return [0, 1]

        # Classify types: True for S-type, False for L-type
        t = [False] * n

        # Sentinel terminal is always S-type
        t[n - 1] = True
        for i in range(n - 2, -1, -1):
            if A[i] < A[i + 1]:
                t[i] = True

            elif A[i] > A[i + 1]:
                t[i] = False

            else:
                t[i] = t[i + 1]


        # Check if an indexis Leftmost S-type (LMS)
        def is_lms(i):
            return i > 0 and t[i] and not t[i - 1]


        # Compute bucket sizes and boundaries
        buckets = [0] * alphabet_size
        for x in A:
            buckets[x] += 1


        def get_buckets():
            head = [0] * alphabet_size
            tail = [0] * alphabet_size
            acc = 0
            for i, count in enumerate(buckets):
                head[i] = acc
                acc += count
                tail[i] = acc

            return head, tail


        # The induced Sorting core pass
        def induce(lms_indices):
            sa = [-1] * n
            head, tail = get_buckets()

            # Place LMS elements at the tails of their buckets
            for idx in reversed(lms_indices):
                c = A[idx]
                tail[c] -= 1
                sa[tail[c]] = idx

            # Induce L-type elements from left to right
            head, tail = get_buckets()
            for i in range(n):
                if sa[i] > 0 and not t[sa[i] - 1]:
                    c = A[sa[i] - 1]
                    sa[head[c]] = sa[i] - 1
                    head[c] += 1

            # Induce S-type elements from right to left
            head, tail = get_buckets()
            for i in range(n - 1, -1, -1):
                if sa[i] > 0 and t[sa[i] - 1]:
                    c = A[sa[i] - 1]
                    tail[c] -= 1
                    sa[tail[c]] = sa[i] - 1

            return sa


        # Collect all raw LMS positions
        lms_positions = [i for i in range(n) if is_lms(i)]

        # Initial guess of LMS sorting to see if there are naming duplicates
        guessed_sa = induce(lms_positions)

        # Filter guessed_sa to find sorted LMS suffixes
        sorted_lms = [idx for idx in guessed_sa if is_lms(idx)]

        # Name/Label the LMS substrings uniquely
        lms_names = [-1] * n
        name_counter = 0
        if sorted_lms:
            lms_names[sorted_lms[0]] = name_counter

        for i in range(1, len(sorted_lms)):
            u, v = sorted_lms[i - 1], sorted_lms[i]
            is_equal = True
            for d in range(n):
                # Check if LMS substrings differ or if we hit the next LMS boundaries
                if A[u + d] != A[v + d] or t[u + d] != t[v + d]:
                    is_equal = False
                    break

                if d > 0 and (is_lms(u + d) or is_lms(v + d)):
                    break

            if not is_equal:
                name_counter += 1

            lms_names[v] = name_counter

        # Form a summary string out of the labels
        summary_A = [lms_names[i] for i in lms_positions]
        summary_alphabet_size = name_counter + 1

        # Recursive Call Rule:
        # If every LMS block got a unique name, sorting is complete.
        # Otherwise, recurse to sort the summary string.
        if summary_alphabet_size < len(summary_A):
            summary_sa = self.sais(summary_A, summary_alphabet_size)

        else:
            summary_sa = [-1] * len(summary_A)
            for i, name in enumerate(summary_A):
                summary_sa[name] = i

        # map back to find the exact, absolute
        ordered_lms = [lms_positions[summary_sa[i]] for i in range(len(summary_A))]

        return induce(ordered_lms)

In [7]:
solution = Solution()

In [8]:
s = "banana"

print(solution.longestDupSubstring(s))

ana


In [9]:
s = "abcd"

print(solution.longestDupSubstring(s))